# meteo.lt API – Meteorologinių duomenų analizė

Šis užrašų knygelis atlieka visus užduoties punktus:
1. Duomenų nuskaitymas per `MeteoClient` klasę
2. Statistinė analizė (metiniai vidurkiai, dienos/nakties temperatūros, lietingi savaitgaliai)
3. Kombinuotas grafikas (paskutinė savaitė + prognozė)
4. Temperatūros persemplavimas iki 5 minučių

In [ ]:
import logging
from datetime import date, timedelta

import matplotlib.pyplot as plt
import pandas as pd

from meteo_client import MeteoClient
from analysis import (
    compute_annual_averages,
    compute_day_night_temperatures,
    count_rainy_weekends,
    plot_temperature_combined,
    resample_to_5min,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Duomenų nuskaitymas

In [ ]:
STATION_CODE = "vilnius"
BASE_URL = "https://api.meteo.lt/v1"

TODAY = date.today()
ONE_YEAR_AGO = TODAY - timedelta(days=365)

client = MeteoClient(station_code=STATION_CODE, base_url=BASE_URL)
print(f"Stoties kodas: {STATION_CODE}")
print(f"Laikotarpis: {ONE_YEAR_AGO} → {TODAY}")

In [ ]:
# 1a) Istoriniai duomenys
df_hist = client.fetch_historical(start=str(ONE_YEAR_AGO), end=str(TODAY))
print(f"Istorinių įrašų: {len(df_hist)}")
df_hist.head()

In [ ]:
# 1b) Prognozės duomenys
df_fore = client.fetch_forecast()
print(f"Prognozės įrašų: {len(df_fore)}")
df_fore.head()

## 2a. Vidutinė metų temperatūra ir oro drėgmė

In [ ]:
averages = compute_annual_averages(df_hist)
print("Metiniai vidurkiai:")
for key, val in averages.items():
    label = "Temperatūra" if "temperature" in key else "Oro drėgmė"
    unit = "°C" if "temperature" in key else "%"
    print(f"  {label}: {val} {unit}")

## 2b. Vidutinė dienos ir nakties temperatūra

In [ ]:
day_night = compute_day_night_temperatures(df_hist)
print("Dienos (08:00–20:00) ir nakties temperatūros:")
for key, val in day_night.items():
    label = "Dienos temp." if "day" in key else "Nakties temp."
    print(f"  {label}: {val} °C")

## 2c. Savaitgaliai su lietaus prognoze

In [ ]:
rainy_count = count_rainy_weekends(df_hist)
print(f"Savaitgalių su prognozuojamu lietumi: {rainy_count}")

## 3. Kombinuotas grafikas: paskutinė savaitė + prognozė

In [ ]:
fig = plot_temperature_combined(df_hist, df_fore, output_path="temperature_combined.png")
plt.show()

## 4. Temperatūros persemplavimas iki 5 minučių

In [ ]:
temp_col = next(
    (c for c in ("airTemperature", "temperature", "temp") if c in df_hist.columns),
    None,
)

if temp_col:
    # Pavyzdžiui naudojame paskutines 24 valandas
    sample = df_hist[temp_col].last("1D")
    resampled = resample_to_5min(sample)

    fig2, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    axes[0].step(sample.index, sample.values, where="post", color="#2196F3", label="Valandiniai duomenys")
    axes[0].set_title("Pradiniai valandiniai matavimai")
    axes[0].set_ylabel("Temperatūra (°C)")
    axes[0].legend()

    axes[1].plot(resampled.index, resampled.values, color="#E91E63", linewidth=0.8, label="5 min interpoliacija")
    axes[1].set_title("Po persemplavimo (5 min, tiesinė interpoliacija)")
    axes[1].set_ylabel("Temperatūra (°C)")
    axes[1].legend()

    fig2.autofmt_xdate()
    plt.tight_layout()
    plt.show()

    print(f"\nPradiniai taškai: {len(sample)}")
    print(f"Po persemplavimo: {len(resampled)}")
    print(resampled.head(14).to_string())
else:
    print("Temperatūros stulpelis nerastas.")